# Cleaning: NYC Collisions

**Author:** Yougi Jain
**Project:** NYC-Collisions

The cleaning steps this notebook used to perform inline now live in
`scripts/clean.py`, so the scheduled refresh workflow and this notebook
apply exactly the same transforms. This notebook explores what `clean()`
does to the raw data.


### Setup

In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path("..").resolve() / "scripts"))
from clean import CANONICAL_COLUMNS, clean, normalize_columns

pd.set_option("display.max_columns", None)

### Raw data

`data/raw/nyc_collisions_sample.csv` is a 5% sample of a manual CSV export,
kept as a small offline fixture. The live pipeline fetches from the Socrata
API instead, which uses different column names; `clean()` handles both.

In [ ]:
raw = pd.read_csv("../data/raw/nyc_collisions_sample.csv")
print(raw.shape)
raw.head()

### Column normalisation

Headers are stripped of punctuation, snake_cased and lowercased. The API's
`vehicle_type_code1` / `vehicle_type_code2` are also reconciled with its own
`vehicle_type_code_3` / `_4` / `_5`.

In [ ]:
normalize_columns(raw).columns.tolist()

### Cleaning

`clean()` builds `crash_datetime` from the separate date and time columns,
coerces types, masks coordinates outside NYC (the source encodes unknown
positions as `0.0`), drops the >98%-null vehicle 4/5 columns and the
redundant `location`, and de-duplicates on `collision_id`.

In [ ]:
cleaned = clean(raw)
print(cleaned.shape)
cleaned.head()

In [ ]:
assert list(cleaned.columns) == CANONICAL_COLUMNS
assert cleaned["crash_datetime"].notna().all()
assert not cleaned["collision_id"].duplicated().any()

print("rows:      ", f"{len(cleaned):,}")
print("date range:", cleaned["crash_datetime"].min(), "->", cleaned["crash_datetime"].max())
print("masked coords:", int(cleaned["latitude"].isna().sum() - raw["LATITUDE"].isna().sum()))

### Building the real dataset

This notebook stops at the sample. To build the full 2020-present dataset
from the API and write the Parquet the dashboard reads:

```bash
python scripts/build_dataset.py --full
```
